# Portable Plot Notebook

This notebook is self-contained for plotting. It does **not** depend on:
- `evaluate_stage1(...)`
- `evaluate_stage2(...)`
- `evaluate_stage3(...)`
- `sample_stage3_latents(...)`
- `write_stage3_metric_trend_plot(...)`
- `extract_current_model_report.py`

The notebook only assumes the codebase still has:
- `diffusion_model/model.py`
- `diffusion_model/dataset.py`
- `diffusion_model/gait_metrics.py`
- `diffusion_model/util.py`

## Required files

### For evaluation / inference plots
- dataset source: either `dataset_path` or all of `skeleton_folder`, `hip_folder`, `wrist_folder`
- `stage1_ckpt` for Stage 1 plots
- `stage1_ckpt` + `stage2_ckpt` for Stage 2 plots
- `stage1_ckpt` + `stage2_ckpt` + `stage3_ckpt` for Stage 3 plots

### For training plots
- `stage1_history`
- `stage2_history`
- `stage3_history`
- `stage3_metric_history`

## Plot categories

### Training
- Stage 1 loss curves
- Stage 2 loss curves
- Stage 3 loss curves
- Stage 3 gait metric trends across epochs

### Evaluation
- Stage 1 noise prediction error
- Stage 1 latent PCA and latent heatmaps
- Stage 2 latent/sensor PCA, t-SNE, UMAP, centroid similarity
- Stage 3 real-vs-generated gait distributions and scatter plots
- Stage 3 conditioning sensitivity
- Stage 3 intermediate diffusion states

### Inference / generation
- generated motion panels
- generated motion GIFs


In [ ]:
from pathlib import Path

CFG = {
    'output_dir': Path(''),
    'dataset_path':'',
    'skeleton_folder':' ',
    'hip_folder': '',
    'wrist_folder': '',
    'gait_cache_dir': '',
    'disable_gait_cache': False,
    'disable_sensor_norm': False,
    'stage1_ckpt': '',
    'stage2_ckpt': '',
    'stage3_ckpt': '',
    'stage1_history': '',
    'stage2_history': '',
    'stage3_history': '',
    'stage3_metric_history': '',
    'window': 90,
    'stride': 45,
    'joints': 32,
    'latent_dim': 256,
    'timesteps': 500,
    'gait_metrics_dim': 9,
    'num_classes': 14,
    'batch_size': 16,
    'fps': 30.0,
    'sample_steps': 50,
    'sampler': 'ddim',
    'num_workers': 0,
    'device': 'cuda',
    'max_stage1_batches': 4,
    'max_stage2_batches': 8,
    'max_stage3_batches': 4,
}

CFG
CFG['dataset_path'] = ''
CFG['skeleton_folder'] = ''
CFG['hip_folder'] = ''
CFG['wrist_folder'] = ''
CFG['device'] = ''


In [151]:
import csv
import json
import math
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw


from matplotlib import pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

from diffusion_model.dataset import create_dataset, create_dataloader
from diffusion_model.gait_metrics import GAIT_METRIC_NAMES, compute_gait_metrics_torch
from diffusion_model.model import Stage1Model, Stage2Model, Stage3Model
from diffusion_model.util import get_skeleton_edges, get_joint_labels


In [ ]:
def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def load_checkpoint_local(path, model, strict=True):
    checkpoint = torch.load(path, map_location='cpu')
    state_dict = checkpoint['state_dict'] if isinstance(checkpoint, dict) and 'state_dict' in checkpoint else checkpoint
    missing, unexpected = model.load_state_dict(state_dict, strict=strict)
    print(f'Loaded checkpoint: {path}')
    if missing:
        print('missing keys:', missing)
    if unexpected:
        print('unexpected keys:', unexpected)
    return checkpoint


def read_history_csv(path):
    with open(path, 'r', newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))


def require_dataset_source(cfg):
    if cfg['dataset_path']:
        return
    if cfg['skeleton_folder'] and cfg['hip_folder'] and cfg['wrist_folder']:
        return
    raise ValueError('Provide either dataset_path or skeleton_folder + hip_folder + wrist_folder.')


def build_dataset_and_loader(cfg):
    require_dataset_source(cfg)
    dataset = create_dataset(
        dataset_path=cfg['dataset_path'] or None,
        window=cfg['window'],
        joints=cfg['joints'],
        num_classes=cfg['num_classes'],
        skeleton_folder=cfg['skeleton_folder'] or None,
        hip_folder=cfg['hip_folder'] or None,
        wrist_folder=cfg['wrist_folder'] or None,
        stride=cfg['stride'],
        normalize_sensors=not cfg['disable_sensor_norm'],
        gait_cache_dir=cfg['gait_cache_dir'] or None,
        disable_gait_cache=cfg['disable_gait_cache'],
    )
    loader = create_dataloader(
        dataset_path=cfg['dataset_path'] or None,
        batch_size=cfg['batch_size'],
        shuffle=False,
        window=cfg['window'],
        joints=cfg['joints'],
        num_classes=cfg['num_classes'],
        skeleton_folder=cfg['skeleton_folder'] or None,
        hip_folder=cfg['hip_folder'] or None,
        wrist_folder=cfg['wrist_folder'] or None,
        stride=cfg['stride'],
        normalize_sensors=not cfg['disable_sensor_norm'],
        gait_cache_dir=cfg['gait_cache_dir'] or None,
        disable_gait_cache=cfg['disable_gait_cache'],
        num_workers=cfg['num_workers'],
        drop_last=False,
    )
    return dataset, loader


def write_curve_plot(out_path, title, x_values, series, x_label, y_label):
    if plt is None:
        return
    ensure_dir(Path(out_path).parent)
    plt.figure(figsize=(12, 7))
    for label, values, color in series:
        plt.plot(x_values, values, label=label, linewidth=2.5, color=color)
    plt.title(title)
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


def write_hist_grid(out_path, title, real_data, gen_data, metric_names):
    if plt is None:
        return
    cols = 3
    rows = int(np.ceil(len(metric_names) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(14, 4.5 * rows))
    axes = np.atleast_1d(axes).reshape(rows, cols)
    for idx, name in enumerate(metric_names):
        ax = axes[idx // cols, idx % cols]
        ax.hist(real_data[:, idx], bins=24, alpha=0.55, color='#2563eb', label='Real', density=True)
        ax.hist(gen_data[:, idx], bins=24, alpha=0.55, color='#dc2626', label='Generated', density=True)
        ax.set_title(name)
        ax.grid(True, alpha=0.2)
    for idx in range(len(metric_names), rows * cols):
        axes[idx // cols, idx % cols].axis('off')
    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='upper right')
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def write_scatter(out_path, title, x_real, y_real, x_gen, y_gen, x_label, y_label):
    if plt is None:
        return
    plt.figure(figsize=(9, 7))
    plt.scatter(x_real, y_real, s=18, alpha=0.45, label='Real', color='#2563eb')
    plt.scatter(x_gen, y_gen, s=18, alpha=0.45, label='Generated', color='#dc2626')
    plt.title(title)
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


def write_heatmap(out_path, title, values, x_label, y_label, cmap='viridis'):
    if plt is None:
        return
    arr = np.asarray(values, dtype=np.float32)
    plt.figure(figsize=(11, 5.5))
    plt.imshow(arr, aspect='auto', cmap=cmap, interpolation='nearest')
    plt.title(title)
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


def write_projection_plot(out_path, title, features, labels, method, num_classes):
    if plt is None or features.shape[0] < 2:
        return False
    method_name = method.lower()
    if method_name == 'pca':
        if PCA is None:
            return False
        reducer = PCA(n_components=2, random_state=42)
        reduced = reducer.fit_transform(features)
        x_label = f"PC1 ({reducer.explained_variance_ratio_[0] * 100:.1f}% var)"
        y_label = f"PC2 ({reducer.explained_variance_ratio_[1] * 100:.1f}% var)"
    elif method_name == 'tsne':
        if TSNE is None or features.shape[0] < 4:
            return False
        perplexity = min(30.0, max(2.0, float(features.shape[0] - 1) / 3.0))
        perplexity = min(perplexity, float(features.shape[0]) - 1.0)
        reducer = TSNE(n_components=2, init='pca', learning_rate='auto', perplexity=perplexity, random_state=42)
        reduced = reducer.fit_transform(features)
        x_label = 't-SNE 1'
        y_label = 't-SNE 2'
    elif method_name == 'umap':
        if umap is None or features.shape[0] < 3:
            return False
        n_neighbors = min(15, features.shape[0] - 1)
        reducer = umap.UMAP(n_components=2, n_neighbors=n_neighbors, min_dist=0.15, random_state=42)
        reduced = reducer.fit_transform(features)
        x_label = 'UMAP 1'
        y_label = 'UMAP 2'
    else:
        raise ValueError(f'Unsupported projection method: {method}')
    plt.figure(figsize=(9, 7))
    cmap = plt.cm.get_cmap('tab20', num_classes)
    for cid in sorted(set(labels.tolist())):
        mask = labels == cid
        plt.scatter(reduced[mask, 0], reduced[mask, 1], s=20, alpha=0.65, color=cmap(cid), label=f'A{cid + 1:02d}')
    plt.title(title)
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.grid(True, alpha=0.25)
    plt.legend(ncol=2, fontsize=9)
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()
    return True


def write_similarity_heatmap(out_path, title, matrix, labels):
    if plt is None or matrix.size == 0:
        return
    fig, ax = plt.subplots(figsize=(8.5, 7))
    image = ax.imshow(matrix, cmap='viridis', aspect='auto')
    ax.set_title(title)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticklabels(labels)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def render_skeleton_panels(out_path, sequences, titles):
    if plt is None or not sequences:
        return
    ensure_dir(Path(out_path).parent)
    edges = [(i, j) for i, j in get_skeleton_edges() if i < sequences[0].shape[1] and j < sequences[0].shape[1]]
    n = len(sequences)
    fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 5.5))
    axes = np.atleast_1d(axes)
    for ax, seq, title in zip(axes, sequences, titles):
        frame = seq[len(seq) // 2]
        xs = frame[:, 0]
        ys = frame[:, 1]
        for i, j in edges:
            ax.plot([xs[i], xs[j]], [ys[i], ys[j]], color='#222', linewidth=1.6)
        ax.scatter(xs, ys, s=20, color='#2563eb')
        ax.set_title(title)
        ax.set_aspect('equal')
        ax.invert_yaxis()
        ax.axis('off')
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def _normalize_xy(points_xy, canvas_size, root_index=0):
    anchor = points_xy[:1, root_index : root_index + 1, :]
    centered = points_xy - anchor
    flat_xy = centered.reshape(-1, 2)
    min_xy = np.percentile(flat_xy, 2.0, axis=0, keepdims=True).reshape(1, 1, 2)
    max_xy = np.percentile(flat_xy, 98.0, axis=0, keepdims=True).reshape(1, 1, 2)
    span = float(np.maximum(max_xy - min_xy, 1e-6).max())
    normalized = (centered - (min_xy + max_xy) * 0.5) / span
    margin = 0.1 * canvas_size
    drawable = canvas_size - 2 * margin
    canvas_center = canvas_size * 0.5
    return normalized * drawable + canvas_center


def save_skeleton_gif(sequence, out_path, fps=12, canvas_size=512):
    ensure_dir(Path(out_path).parent)
    points = np.asarray(sequence, dtype=np.float32)
    points_xy = _normalize_xy(points[..., :2], canvas_size=canvas_size)
    edges = [(i, j) for i, j in get_skeleton_edges() if i < points.shape[1] and j < points.shape[1]]
    frames = []
    for frame_xy in points_xy:
        img = Image.new('RGB', (canvas_size, canvas_size), color=(255, 255, 255))
        draw = ImageDraw.Draw(img)
        for i, j in edges:
            xi, yi = float(frame_xy[i][0]), float(frame_xy[i][1])
            xj, yj = float(frame_xy[j][0]), float(frame_xy[j][1])
            draw.line((xi, yi, xj, yj), fill=(30, 30, 30), width=3)
        for joint_xy in frame_xy:
            x, y = float(joint_xy[0]), float(joint_xy[1])
            r = 4
            draw.ellipse((x - r, y - r, x + r, y + r), fill=(20, 60, 210))
        frames.append(img)
    duration_ms = int(1000 / max(fps, 1))
    frames[0].save(out_path, save_all=True, append_images=frames[1:], duration=duration_ms, loop=0)


def _snapshot_timesteps(total_timesteps):
    if total_timesteps < 2:
        return [0]
    last = total_timesteps - 1
    candidates = [last, int(round(last * 0.6)), int(round(last * 0.2)), 0]
    ordered = []
    for item in candidates:
        item = max(0, min(last, int(item)))
        if item not in ordered:
            ordered.append(item)
    return ordered


def write_metric_relationship_grid(out_path, real_data, generated_data, metric_names, pair_indices):
    if plt is None or not pair_indices:
        return
    cols = 2
    rows = math.ceil(len(pair_indices) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(14, 5.5 * rows))
    axes_arr = np.array(axes).reshape(-1)
    for ax, (i, j) in zip(axes_arr, pair_indices):
        ax.scatter(real_data[:, i], real_data[:, j], s=16, alpha=0.4, color='#2563eb', label='Real')
        ax.scatter(generated_data[:, i], generated_data[:, j], s=16, alpha=0.4, color='#dc2626', label='Generated')
        ax.set_xlabel(metric_names[i])
        ax.set_ylabel(metric_names[j])
        ax.grid(True, alpha=0.2)
        ax.set_title(f'{metric_names[j]} vs {metric_names[i]}')
    for idx in range(len(pair_indices), len(axes_arr)):
        axes_arr[idx].axis('off')
    handles, labels = axes_arr[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='upper right')
    fig.suptitle('Real vs Generated Metric Relationships')
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def write_label_distribution(out_path, counts, num_classes):
    if plt is None:
        return
    labels = [f'A{i+1:02d}' for i in range(num_classes)]
    values = [counts.get(i, 0) for i in range(num_classes)]
    plt.figure(figsize=(12, 6))
    plt.bar(labels, values, color='#2563eb')
    plt.title('Real Dataset Window Distribution by Activity')
    plt.xlabel('Activity label')
    plt.ylabel('Window count')
    plt.grid(True, axis='y', alpha=0.25)
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


In [153]:
def sample_stage3_latents_local(stage3, shape, device, h_tokens, h_global, gait_metrics, sample_steps, sampler='ddim'):
    sampler_name = sampler.lower()
    if sampler_name == 'ddim':
        return stage3.diffusion.p_sample_loop_ddim(
            stage3.denoiser,
            shape=shape,
            device=device,
            sample_steps=sample_steps,
            eta=0.0,
            h_tokens=h_tokens,
            h_global=h_global,
            gait_metrics=gait_metrics,
        )
    if sampler_name == 'ddpm':
        return stage3.diffusion.p_sample_loop(
            stage3.denoiser,
            shape=shape,
            device=device,
            h_tokens=h_tokens,
            h_global=h_global,
            gait_metrics=gait_metrics,
        )
    raise ValueError(f'Unsupported sampler: {sampler}')


def evaluate_stage1_local(model, loader, device, out_dir, timestep_values, max_batches):
    rows = []
    latent_features = []
    labels = []
    sample_sequences = []
    sample_latent_maps = []
    with torch.no_grad():
        for t_val in timestep_values:
            vals = []
            for batch_idx, batch in enumerate(loader):
                if batch_idx >= max_batches:
                    break
                x = batch['skeleton'].to(device)
                gait_metrics = batch['gait_metrics'].to(device)
                y = batch['label'].cpu().numpy()
                z0 = model.encoder(x, gait_metrics=gait_metrics)
                if len(sample_sequences) < 3:
                    remaining = 3 - len(sample_sequences)
                    for seq, latent, label in zip(x[:remaining], z0[:remaining], y[:remaining]):
                        sample_sequences.append(seq.cpu().numpy())
                        latent_map = torch.linalg.norm(latent.float(), dim=-1).cpu().numpy()
                        sample_latent_maps.append((latent_map, f'label_A{int(label) + 1:02d}'))
                latent_features.append(z0.mean(dim=(1, 2)).cpu().numpy())
                labels.append(y)
                t = torch.full((x.shape[0],), int(t_val), device=device, dtype=torch.long)
                noise = torch.randn_like(z0)
                zt = model.diffusion.q_sample(z0=z0, t=t, noise=noise)
                pred_noise = model.denoiser(zt, t, gait_metrics=gait_metrics)
                mse = torch.mean((pred_noise - noise) ** 2, dim=(1, 2, 3)).cpu().numpy()
                vals.extend(mse.tolist())
            if vals:
                rows.append({'timestep': int(t_val), 'mean_mse': float(np.mean(vals)), 'std_mse': float(np.std(vals)), 'count': len(vals)})
    ensure_dir(out_dir)
    if rows:
        write_curve_plot(
            Path(out_dir) / 'noise_prediction_error_by_timestep.png',
            'Noise Prediction Error by Diffusion Timestep',
            [row['timestep'] for row in rows],
            [('Mean MSE', [row['mean_mse'] for row in rows], '#2563eb')],
            'Diffusion timestep',
            'MSE(pred_noise, true_noise)',
        )
    if sample_sequences:
        render_skeleton_panels(Path(out_dir) / 'encoder_input_skeletons.png', sample_sequences, [f'Input sample {idx}' for idx in range(len(sample_sequences))])
    for idx, (latent_map, label_text) in enumerate(sample_latent_maps):
        write_heatmap(Path(out_dir) / f'encoder_output_latent_norm_sample_{idx}.png', f'Stage1 Encoder Output Norm Map ({label_text})', latent_map.T, 'Frame', 'Joint', cmap='magma')
    if latent_features and labels:
        latent_arr = np.concatenate(latent_features, axis=0)
        label_arr = np.concatenate(labels, axis=0)
        write_projection_plot(Path(out_dir) / 'encoder_output_latent_pca.png', 'Stage1 Encoder Output PCA (mean pooled z0)', latent_arr, label_arr, method='pca', num_classes=CFG['num_classes'])


def _class_centroids(features, labels):
    classes = sorted({int(cid) for cid in labels.tolist()})
    centroids = []
    for cid in classes:
        class_features = features[labels == cid]
        centroids.append(class_features.mean(axis=0))
    return np.asarray(centroids, dtype=np.float32), classes


def evaluate_stage2_local(stage1, stage2, loader, device, out_dir, max_batches):
    latent_features = []
    sensor_features = []
    labels = []
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            if batch_idx >= max_batches:
                break
            x = batch['skeleton'].to(device)
            a_hip = batch['A_hip'].to(device)
            a_wrist = batch['A_wrist'].to(device)
            gait_metrics = batch['gait_metrics'].to(device)
            y = batch['label'].cpu().numpy()
            z0 = stage1.encoder(x, gait_metrics=gait_metrics)
            _, h_global = stage2.aligner(a_hip, a_wrist, gait_metrics=gait_metrics)
            latent_features.append(z0.mean(dim=(1, 2)).cpu().numpy())
            sensor_features.append(h_global.cpu().numpy())
            labels.append(y)
    ensure_dir(out_dir)
    if not latent_features:
        return
    latent_arr = np.concatenate(latent_features, axis=0)
    sensor_arr = np.concatenate(sensor_features, axis=0)
    label_arr = np.concatenate(labels, axis=0)
    for method in ['pca', 'tsne', 'umap']:
        write_projection_plot(Path(out_dir) / f'latent_{method}.png', f'Skeleton Latent Space {method.upper()}', latent_arr, label_arr, method=method, num_classes=CFG['num_classes'])
        write_projection_plot(Path(out_dir) / f'sensor_embedding_{method}.png', f'Sensor Embedding {method.upper()}', sensor_arr, label_arr, method=method, num_classes=CFG['num_classes'])
    write_projection_plot(Path(out_dir) / 'skeleton_latent_umap.png', 'Skeleton Latent Space UMAP', latent_arr, label_arr, method='umap', num_classes=CFG['num_classes'])
    write_projection_plot(Path(out_dir) / 'sensor_embedding_umap_explicit.png', 'Sensor Embedding UMAP', sensor_arr, label_arr, method='umap', num_classes=CFG['num_classes'])
    latent_centroids, centroid_classes = _class_centroids(latent_arr, label_arr)
    sensor_centroids, _ = _class_centroids(sensor_arr, label_arr)
    centroid_labels = [f'A{cid + 1:02d}' for cid in centroid_classes]
    latent_centroid_sim = F.cosine_similarity(torch.from_numpy(latent_centroids)[:, None, :], torch.from_numpy(latent_centroids)[None, :, :], dim=-1).cpu().numpy()
    sensor_centroid_sim = F.cosine_similarity(torch.from_numpy(sensor_centroids)[:, None, :], torch.from_numpy(sensor_centroids)[None, :, :], dim=-1).cpu().numpy()
    write_similarity_heatmap(Path(out_dir) / 'latent_class_centroid_similarity.png', 'Skeleton Latent Class-Centroid Cosine Similarity', latent_centroid_sim, centroid_labels)
    write_similarity_heatmap(Path(out_dir) / 'sensor_class_centroid_similarity.png', 'Sensor Embedding Class-Centroid Cosine Similarity', sensor_centroid_sim, centroid_labels)


def write_stage3_metric_trend_plot_local(out_path, rows):
    if plt is None or not rows:
        return
    metric_names = list(GAIT_METRIC_NAMES)
    cols = 3
    plot_rows = int(np.ceil(len(metric_names) / cols))
    fig, axes = plt.subplots(plot_rows, cols, figsize=(15, 4.8 * plot_rows))
    axes = np.atleast_1d(axes).reshape(plot_rows, cols)
    for idx, metric_name in enumerate(metric_names):
        ax = axes[idx // cols, idx % cols]
        metric_rows = sorted([row for row in rows if str(row['metric_name']) == metric_name], key=lambda row: int(float(row['epoch'])))
        epochs = [int(float(row['epoch'])) for row in metric_rows]
        gen_mean = [float(row['generated_mean']) for row in metric_rows]
        gen_std = [float(row['generated_std']) for row in metric_rows]
        real_mean = [float(row['real_mean']) for row in metric_rows]
        real_std = [float(row['real_std']) for row in metric_rows]
        ax.plot(epochs, gen_mean, color='#dc2626', linewidth=2.0, label='Generated mean')
        ax.plot(epochs, gen_std, color='#f97316', linewidth=2.0, label='Generated std')
        ax.plot(epochs, real_mean, color='#2563eb', linewidth=1.5, linestyle='--', label='Real mean')
        ax.plot(epochs, real_std, color='#10b981', linewidth=1.5, linestyle='--', label='Real std')
        ax.set_title(metric_name)
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.25)
    for idx in range(len(metric_names), plot_rows * cols):
        axes[idx // cols, idx % cols].axis('off')
    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='upper right')
    fig.suptitle('Generated Gait Metric Mean/Std Across Evaluation Epochs')
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def evaluate_stage3_local(stage2, stage3, loader, device, out_dir, sample_steps, fps, sampler, max_batches):
    ensure_dir(out_dir)
    real_gait_list = []
    gen_gait_list = []
    gen_sequences = []
    conditioning_rows = []
    snapshot_written = False
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            if batch_idx >= max_batches:
                break
            x = batch['skeleton'].to(device)
            y = batch['label'].to(device)
            a_hip = batch['A_hip'].to(device)
            a_wrist = batch['A_wrist'].to(device)
            gait_metrics = batch['gait_metrics'].to(device)
            h_tokens, h_global = stage2.aligner(a_hip, a_wrist, gait_metrics=gait_metrics)
            cond_tokens, cond_global = stage3.condition_with_labels(h_tokens=h_tokens, h_global=h_global, y=y)
            shape = (x.shape[0], x.shape[1], x.shape[2], stage3.latent_dim)
            z0_gen = sample_stage3_latents_local(stage3, torch.Size(shape), device, cond_tokens, cond_global, gait_metrics, sample_steps, sampler=sampler)
            x_hat = stage3.decoder(z0_gen)
            gait_gen = compute_gait_metrics_torch(x_hat, fps=fps).cpu().numpy()
            real_gait_list.append(gait_metrics.cpu().numpy())
            gen_gait_list.append(gait_gen)
            if len(gen_sequences) < 3:
                gen_sequences.extend([seq for seq in x_hat.cpu().numpy()[: 3 - len(gen_sequences)]])
            if not snapshot_written:
                sample_gait = gait_metrics[:1]
                sample_y = y[:1]
                sample_hip = a_hip[:1]
                sample_wrist = a_wrist[:1]
                s_tokens, s_global = stage2.aligner(sample_hip, sample_wrist, gait_metrics=sample_gait)
                c_tokens, c_global = stage3.condition_with_labels(h_tokens=s_tokens, h_global=s_global, y=sample_y)
                z = torch.randn((1, x.shape[1], x.shape[2], stage3.latent_dim), device=device)
                captured = {}
                capture_steps = _snapshot_timesteps(stage3.diffusion.timesteps)
                for i in reversed(range(stage3.diffusion.timesteps)):
                    t = torch.full((1,), i, device=device, dtype=torch.long)
                    pred_noise = stage3.denoiser(z, t, h_tokens=c_tokens, h_global=c_global, gait_metrics=sample_gait)
                    alpha_bar_t = stage3.diffusion.alphas_cumprod[i]
                    sqrt_alpha_bar_t = torch.sqrt(torch.clamp(alpha_bar_t, min=1e-20))
                    sqrt_one_minus = torch.sqrt(torch.clamp(1.0 - alpha_bar_t, min=1e-20))
                    x0_pred = (z - sqrt_one_minus * pred_noise) / sqrt_alpha_bar_t
                    if i in capture_steps:
                        captured[i] = stage3.decoder(x0_pred).cpu().numpy()[0]
                    z = stage3.diffusion.p_sample(stage3.denoiser, z, t, h_tokens=c_tokens, h_global=c_global, gait_metrics=sample_gait)
                times = capture_steps
                render_skeleton_panels(Path(out_dir) / 'intermediate_diffusion_states.png', [captured[t] for t in times if t in captured], [f't={t}' for t in times if t in captured])
                snapshot_written = True
            if batch_idx == 0 and len(loader.dataset) >= 2:
                sample_a = loader.dataset[0]
                sample_b = loader.dataset[min(100, len(loader.dataset) - 1)]
                seed = 123
                a_hip_fixed = sample_a['A_hip'].unsqueeze(0).to(device)
                a_wrist_fixed = sample_a['A_wrist'].unsqueeze(0).to(device)
                x_fixed = sample_a['skeleton'].unsqueeze(0).to(device)
                gait_a = sample_a['gait_metrics'].unsqueeze(0).to(device)
                gait_b = sample_b['gait_metrics'].unsqueeze(0).to(device)
                y_fixed = sample_a['label'].view(1).to(device)
                def _generate(gait_tensor):
                    torch.manual_seed(seed)
                    s_tokens, s_global = stage2.aligner(a_hip_fixed, a_wrist_fixed, gait_metrics=gait_tensor)
                    c_tokens, c_global = stage3.condition_with_labels(h_tokens=s_tokens, h_global=s_global, y=y_fixed)
                    z0_local = sample_stage3_latents_local(stage3, torch.Size((1, x_fixed.shape[1], x_fixed.shape[2], stage3.latent_dim)), device, c_tokens, c_global, gait_tensor, sample_steps, sampler=sampler)
                    x_local = stage3.decoder(z0_local)
                    return compute_gait_metrics_torch(x_local, fps=fps).cpu().numpy()[0]
                gen_a = _generate(gait_a)
                gen_b = _generate(gait_b)
                for i, name in enumerate(GAIT_METRIC_NAMES):
                    conditioning_rows.append({'metric_name': name, 'conditioning_a': float(gait_a.cpu().numpy()[0, i]), 'conditioning_b': float(gait_b.cpu().numpy()[0, i]), 'generated_a': float(gen_a[i]), 'generated_b': float(gen_b[i]), 'generated_delta': float(gen_b[i] - gen_a[i])})
    ensure_dir(out_dir)
    if not gen_gait_list:
        return
    real_gait = np.concatenate(real_gait_list, axis=0)
    gen_gait = np.concatenate(gen_gait_list, axis=0)
    write_hist_grid(Path(out_dir) / 'real_vs_generated_gait_distributions.png', 'Real vs Generated Gait Metric Distributions', real_gait, gen_gait, GAIT_METRIC_NAMES)
    write_scatter(Path(out_dir) / 'real_vs_generated_speed_vs_com_fore_aft.png', 'Real vs Generated: Walking Speed vs Mean CoM Fore-Aft', real_gait[:, 6], real_gait[:, 0], gen_gait[:, 6], gen_gait[:, 0], 'Mean Walking Speed', 'Mean CoM Fore-Aft')
    if conditioning_rows and plt is not None:
        plt.figure(figsize=(12, 6))
        plt.bar([row['metric_name'] for row in conditioning_rows], [row['generated_delta'] for row in conditioning_rows], color='#2563eb')
        plt.title('Conditioning Sensitivity Under Fixed Seed')
        plt.xlabel('Gait metric')
        plt.ylabel('Generated metric delta (B - A)')
        plt.xticks(rotation=40, ha='right')
        plt.grid(True, axis='y', alpha=0.25)
        plt.tight_layout()
        plt.savefig(Path(out_dir) / 'conditioning_sensitivity.png', dpi=180)
        plt.close()
    if gen_sequences:
        render_skeleton_panels(Path(out_dir) / 'generated_motion_examples.png', gen_sequences, [f'sample_{i}' for i in range(len(gen_sequences))])
        for idx, seq in enumerate(gen_sequences):
            save_skeleton_gif(seq, Path(out_dir) / f'generated_motion_example_{idx}.gif', fps=max(1, int(round(fps / 2.5))))


In [154]:
def regenerate_all(cfg):
    output_dir = ensure_dir(cfg['output_dir'])
    training_dir = ensure_dir(output_dir / 'training')
    evaluation_dir = ensure_dir(output_dir / 'evaluation')
    reporting_dir = ensure_dir(output_dir / 'reporting')

    device_name = cfg['device']
    if device_name == 'cuda' and not torch.cuda.is_available():
        device_name = 'cpu'
    device = torch.device(device_name)
    print('device:', device)

    if cfg['stage1_history']:
        rows = read_history_csv(cfg['stage1_history'])
        write_curve_plot(training_dir / 'stage1_loss_curves.png', 'Stage1 Loss Curves', [float(r['epoch']) for r in rows], [('train_loss_diff', [float(r['train_loss_diff']) for r in rows], '#2563eb'), ('val_loss_diff', [float(r['val_loss_diff']) for r in rows], '#dc2626')], 'Epoch', 'Loss')

    if cfg['stage2_history']:
        rows = read_history_csv(cfg['stage2_history'])
        write_curve_plot(training_dir / 'stage2_loss_curves.png', 'Stage2 Loss Curves', [float(r['epoch']) for r in rows], [('train_loss_align', [float(r['train_loss_align']) for r in rows], '#2563eb'), ('val_loss_align', [float(r['val_loss_align']) for r in rows], '#dc2626')], 'Epoch', 'Loss')

    if cfg['stage3_history']:
        rows = read_history_csv(cfg['stage3_history'])
        write_curve_plot(training_dir / 'stage3_total_loss_curves.png', 'Stage3 Total Loss Curves', [float(r['epoch']) for r in rows], [('train_loss_total', [float(r['train_loss_total']) for r in rows], '#111827'), ('val_loss_total', [float(r['val_loss_total']) for r in rows], '#7c3aed')], 'Epoch', 'Loss')
        write_curve_plot(training_dir / 'stage3_component_loss_curves.png', 'Stage3 Component Losses', [float(r['epoch']) for r in rows], [('train_loss_diff', [float(r['train_loss_diff']) for r in rows], '#2563eb'), ('train_loss_cls', [float(r['train_loss_cls']) for r in rows], '#dc2626'), ('train_loss_gait', [float(r['train_loss_gait']) for r in rows], '#059669'), ('train_loss_motion', [float(r['train_loss_motion']) for r in rows], '#0f766e')], 'Epoch', 'Loss')

    if cfg['stage3_metric_history']:
        rows = read_history_csv(cfg['stage3_metric_history'])
        write_stage3_metric_trend_plot_local(training_dir / 'generated_gait_metric_trends.png', rows)

    dataset = None
    loader = None
    if cfg['stage1_ckpt'] or cfg['stage2_ckpt'] or cfg['stage3_ckpt']:
        dataset, loader = build_dataset_and_loader(cfg)
        print('dataset windows:', len(dataset))

    stage1 = None
    stage2 = None
    stage3 = None

    if cfg['stage1_ckpt']:
        stage1 = Stage1Model(latent_dim=cfg['latent_dim'], num_joints=cfg['joints'], timesteps=cfg['timesteps'], gait_metrics_dim=cfg['gait_metrics_dim']).to(device)
        load_checkpoint_local(str(cfg['stage1_ckpt']), stage1, strict=True)
        stage1.eval()
        evaluate_stage1_local(stage1, loader, device, evaluation_dir / 'stage1_eval', [0, 50, 100, 200, 300, 400, cfg['timesteps'] - 1], cfg['max_stage1_batches'])

    if cfg['stage1_ckpt'] and cfg['stage2_ckpt']:
        stage2 = Stage2Model(encoder=stage1.encoder, latent_dim=cfg['latent_dim'], num_joints=cfg['joints'], gait_metrics_dim=cfg['gait_metrics_dim']).to(device)
        load_checkpoint_local(str(cfg['stage2_ckpt']), stage2, strict=True)
        stage2.eval()
        evaluate_stage2_local(stage1, stage2, loader, device, evaluation_dir / 'stage2_eval', cfg['max_stage2_batches'])

    if cfg['stage1_ckpt'] and cfg['stage2_ckpt'] and cfg['stage3_ckpt']:
        stage3 = Stage3Model(encoder=stage1.encoder, decoder=stage1.decoder, denoiser=stage1.denoiser, latent_dim=cfg['latent_dim'], num_joints=cfg['joints'], num_classes=cfg['num_classes'], timesteps=cfg['timesteps'], gait_metrics_dim=cfg['gait_metrics_dim']).to(device)
        load_checkpoint_local(str(cfg['stage3_ckpt']), stage3, strict=True)
        stage3.eval()
        evaluate_stage3_local(stage2, stage3, loader, device, evaluation_dir / 'stage3_eval', cfg['sample_steps'], cfg['fps'], cfg['sampler'], cfg['max_stage3_batches'])
        real_gait = []
        gen_gait = []
        with torch.no_grad():
            for batch_idx, batch in enumerate(loader):
                if batch_idx >= cfg['max_stage3_batches']:
                    break
                x = batch['skeleton'].to(device)
                y = batch['label'].to(device)
                a_hip = batch['A_hip'].to(device)
                a_wrist = batch['A_wrist'].to(device)
                gait_metrics = batch['gait_metrics'].to(device)
                h_tokens, h_global = stage2.aligner(a_hip, a_wrist, gait_metrics=gait_metrics)
                cond_tokens, cond_global = stage3.condition_with_labels(h_tokens=h_tokens, h_global=h_global, y=y)
                shape = (x.shape[0], x.shape[1], x.shape[2], stage3.latent_dim)
                z0_gen = sample_stage3_latents_local(stage3, torch.Size(shape), device, cond_tokens, cond_global, gait_metrics, cfg['sample_steps'], sampler=cfg['sampler'])
                x_hat = stage3.decoder(z0_gen)
                gen_gait.append(compute_gait_metrics_torch(x_hat, fps=cfg['fps']).cpu().numpy())
                real_gait.append(gait_metrics.cpu().numpy())
        real_gait = np.concatenate(real_gait, axis=0)
        gen_gait = np.concatenate(gen_gait, axis=0)
        write_metric_relationship_grid(reporting_dir / 'real_vs_generated_metric_relationships.png', real_gait, gen_gait, list(GAIT_METRIC_NAMES), [(6, 0), (6, 4), (6, 7), (6, 8), (0, 4), (7, 8)])
        write_hist_grid(reporting_dir / 'real_vs_generated_gait_distributions.png', 'Real vs Generated Gait Metric Distributions', real_gait, gen_gait, GAIT_METRIC_NAMES)
        write_scatter(reporting_dir / 'real_vs_generated_speed_vs_com_fore_aft.png', 'Real vs Generated: Walking Speed vs Mean CoM Fore-Aft', real_gait[:, 6], real_gait[:, 0], gen_gait[:, 6], gen_gait[:, 0], 'Mean Walking Speed', 'Mean CoM Fore-Aft')

    if dataset is not None:
        label_counter = Counter(int(dataset.label[i].item()) for i in range(len(dataset.label)))
        write_label_distribution(reporting_dir / 'dataset_label_distribution.png', label_counter, cfg['num_classes'])

    print('done')
    print('training plots:', training_dir)
    print('evaluation plots:', evaluation_dir)
    print('reporting plots:', reporting_dir)


## Run

After setting `CFG`, run this cell.


In [155]:
regenerate_all(CFG)


device: cuda:7
[dataset] Skipped invalid paired sample: S30A02T01.csv
[dataset] Skipped invalid paired sample: S30A02T02.csv
[dataset] Skipped invalid paired sample: S30A02T03.csv
[dataset] Skipped invalid paired sample: S30A02T04.csv
[dataset] Skipped invalid paired sample: S30A02T05.csv
[dataset] Skipped invalid paired sample: S30A03T01.csv
[dataset] Skipped invalid paired sample: S30A03T02.csv
[dataset] Skipped invalid paired sample: S30A03T03.csv
[dataset] Skipped invalid paired sample: S30A03T04.csv
[dataset] Skipped invalid paired sample: S30A03T05.csv
[dataset] Skipped invalid paired sample: S30A05T05.csv
[dataset] Skipped invalid paired sample: S30A06T01.csv
[dataset] Skipped invalid paired sample: S30A06T02.csv
[dataset] Skipped invalid paired sample: S30A06T03.csv
[dataset] Skipped invalid paired sample: S30A06T04.csv
[dataset] Skipped invalid paired sample: S30A06T05.csv
[dataset] Skipped invalid paired sample: S30A08T01.csv
[dataset] Skipped invalid paired sample: S30A08T02

/tmp/ipykernel_1133506/435858766.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location='cpu')


Loaded checkpoint: /home/qsw26/smartfall/gait_loss/ldm_gait_loss/outputs/retrain_meeting_march_13/checkpoints/stage1_best.pt


/tmp/ipykernel_1133506/435858766.py:170: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('tab20', num_classes)
/tmp/ipykernel_1133506/435858766.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_s

Loaded checkpoint: /home/qsw26/smartfall/gait_loss/ldm_gait_loss/outputs/retrain_meeting_march_13/checkpoints/stage2_best.pt


/tmp/ipykernel_1133506/435858766.py:170: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('tab20', num_classes)
/tmp/ipykernel_1133506/435858766.py:170: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('tab20', num_classes)
/tmp/ipykernel_1133506/435858766.py:170: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('tab20', num_classes)
/tmp/ipykernel_1133506/435858766.py:170: MatplotlibDeprecationWarning:

Loaded checkpoint: /home/qsw26/smartfall/gait_loss/ldm_gait_loss/outputs/retrain_meeting_march_13/checkpoints/stage3_best.pt
done
training plots: /home/qsw26/smartfall/gait_loss/ldm_gait_loss/outputs/portable_plot_regen/training
evaluation plots: /home/qsw26/smartfall/gait_loss/ldm_gait_loss/outputs/portable_plot_regen/evaluation
reporting plots: /home/qsw26/smartfall/gait_loss/ldm_gait_loss/outputs/portable_plot_regen/reporting
